In [0]:
# Dynamic path resolution (no hardcoded user email)
_user = spark.sql("SELECT current_user()").first()[0]
_base = f"file:/Workspace/Users/{_user}/databricks_ai/MachineLearning"

df = spark.read.format("csv").option("header", "true").load(f"{_base}/diabetes.csv")
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
   

data = df.dropna().select(col("Pregnancies").astype("int"),
                           col("Glucose").astype("int"),
                          col("BloodPressure").astype("int"),
                          col("SkinThickness").astype("int"),
                          col("Insulin").astype("int"),
                          col("BMI").astype("float"),
                          col("DiabetesPedigreeFunction").astype("float"),
                          col("Age").astype("int"),
                          col("Outcome").astype("int")
                          )

   
splits = data.randomSplit([0.7, 0.3])
train = splits[0]
test = splits[1]
print ("Training Rows:", train.count(), " Testing Rows:", test.count())

### Creating an MLflow Experiment Function

In [0]:
def train_diabetes_model(training_data, test_data, maxIterations, regularization):
    import mlflow
    import mlflow.sklearn
    from mlflow.models import infer_signature
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import accuracy_score, recall_score, precision_score
    import pandas as pd
    import time
    
    # Start an MLflow run  
    with mlflow.start_run():
        numFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", 
                       "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

        # Convert Spark DataFrames to pandas
        train_pd = training_data.select(numFeatures + ["Outcome"]).toPandas()
        test_pd = test_data.select(numFeatures + ["Outcome"]).toPandas()
        
        X_train = train_pd[numFeatures]
        y_train = train_pd["Outcome"]
        X_test = test_pd[numFeatures]
        y_test = test_pd["Outcome"]

        # Scale features
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Train scikit-learn Logistic Regression
        print("Training Logistic Regression model...")
        mlflow.log_param('maxIter', maxIterations)
        mlflow.log_param('regParam', regularization)
        
        model = LogisticRegression(max_iter=maxIterations, C=1.0/regularization, random_state=42)
        model.fit(X_train_scaled, y_train)
   
        # Evaluate the model
        y_pred = model.predict(X_test_scaled)
        
        accuracy = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred, average='weighted')
        precision = precision_score(y_test, y_pred, average='weighted')
        
        print("accuracy: %s" % accuracy)
        print("weightedRecall: %s" % recall)
        print("weightedPrecision: %s" % precision)
        
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('weightedRecall', recall)
        mlflow.log_metric('weightedPrecision', precision)
   
        # Create signature
        input_example = pd.DataFrame([{
            "Pregnancies": 8, "Glucose": 85, "BloodPressure": 65, "SkinThickness": 29,
            "Insulin": 0, "BMI": 26.6, "DiabetesPedigreeFunction": 0.672, "Age": 34
        }])
        output_example = pd.DataFrame({"prediction": [1]})
        signature = infer_signature(input_example, output_example)
   
        # Log the sklearn model with signature
        unique_model_name = "diabetes_sklearn_" + str(int(time.time()))
        
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=unique_model_name,
            signature=signature,
            input_example=input_example
        )
   
        print("Experiment run complete with sklearn model.")
     

### Calling our MLflow experiment function with different hyperparameters

In [0]:
train_diabetes_model(train, test, 5, 0.5)

In [0]:
train_diabetes_model(train, test, 10, 0.2)